# BUS 104: Nonlinear Programming Homework Starter Notebook (`NLP_Homework_Starter.ipynb`)
**Companion to the Interactive HTML5 Simulator (`Homework_Files/nlp-sim.html`)**

Use this notebook alongside the interactive simulator (`https://dxt48.github.io/BUS-104/Homework_Files/nlp-sim.html`) to verify all 5 Rounds (Questions 1–30) of the Nonlinear Programming Homework using `scipy.optimize` (`minimize`, `differential_evolution`, and `LinearConstraint`).

In [1]:
import numpy as np
from scipy.optimize import minimize, differential_evolution, LinearConstraint

# ==========================================================================
# ROUND 1 (Q1-Q6): TEST-STEP RULE & TWO-PRODUCT BUNDLE CANNIBALIZATION
# ==========================================================================
print('=== ROUND 1A: SaaS Subscription Tiers (in $100s/day) ===')
tiers = {
    'Tier A (Campus Stream)': (52, 2, 40),
    'Tier B (Cloud Pro)':     (72, 3, 60),
    'Tier C (Enterprise AI)': (96, 3, 110),
}
for name, (a, b, f) in tiers.items():
    res = minimize(lambda x: -(a*x[0] - b*(x[0]**2) - f), x0=[5.0], bounds=[(0, 30)])
    xs = round(float(res.x[0]))
    pi_l = a*(xs-1) - b*((xs-1)**2) - f
    pi_m = a*xs - b*(xs**2) - f
    pi_r = a*(xs+1) - b*((xs+1)**2) - f
    print(f'{name}: x* = ${xs}, Pi(x*-1) = {pi_l}, Pi(x*) = {pi_m} (${pi_m*100:,}/day), Pi(x*+1) = {pi_r}')

print('\n=== ROUND 1B: Two-Product Bundle Cannibalization ===')
def neg_bundle_profit(p):
    p1, p2 = p
    return -(144*p1 + 150*p2 - 4*(p1**2) - 5*(p2**2) - 2*p1*p2 - 200)

res_r1b = minimize(neg_bundle_profit, x0=[10.0, 10.0], bounds=[(0, 40), (0, 40)])
p1_opt, p2_opt = res_r1b.x
print(f'Policy 2 (Joint NLP Optimum): p1* = ${p1_opt:.2f}, p2* = ${p2_opt:.2f}, Max Profit = ${-res_r1b.fun:.2f}/day')
print(f'Policy 1 (Naive Separate p1=$18, p2=$15): Profit = ${-neg_bundle_profit([18, 15]):.2f}/day')
print(f'Policy 3 (Equal Price p1=p2=$14):         Profit = ${-neg_bundle_profit([14, 14]):.2f}/day')

=== ROUND 1A: SaaS Subscription Tiers (in $100s/day) ===
Tier A (Campus Stream): x* = $13, Pi(x*-1) = 296, Pi(x*) = 298 ($29,800/day), Pi(x*+1) = 296
Tier B (Cloud Pro): x* = $12, Pi(x*-1) = 369, Pi(x*) = 372 ($37,200/day), Pi(x*+1) = 369
Tier C (Enterprise AI): x* = $16, Pi(x*-1) = 655, Pi(x*) = 658 ($65,800/day), Pi(x*+1) = 655

=== ROUND 1B: Two-Product Bundle Cannibalization ===
Policy 2 (Joint NLP Optimum): p1* = $15.00, p2* = $12.00, Max Profit = $1780.00/day
Policy 1 (Naive Separate p1=$18, p2=$15): Profit = $1681.00/day
Policy 3 (Equal Price p1=p2=$14):         Profit = $1760.00/day


In [2]:
# ==========================================================================
# ROUND 2 & ROUND 3 (Q7-Q18): BIMODAL LANDSCAPE, MULTI-START & EVOLUTIONARY
# ==========================================================================
def neg_bimodal(x):
    val = x[0]
    return -(78.0 * np.exp(-((val - 28.0)/12.0)**2) + 115.0 * np.exp(-((val - 82.0)/15.0)**2))

print('=== ROUND 2A: Hill-Climbing (minimize) from Different Starting Guesses x0 ===')
for x0_val in [10, 38, 52, 62, 90]:
    r = minimize(neg_bimodal, x0=[x0_val], bounds=[(0, 110)])
    print(f'Start x0 = [{x0_val:2d}] -> Converges to x = {r.x[0]:.2f}, Profit = ${-r.fun:.2f}k')

res_de = differential_evolution(neg_bimodal, bounds=[(0, 110)], seed=42)
print(f'\n=== ROUND 3A: differential_evolution Global Optimum ===')
print(f'Global Peak x* = {res_de.x[0]:.2f}, Global Max Profit = ${-res_de.fun:.2f}k')

=== ROUND 2A: Hill-Climbing (minimize) from Different Starting Guesses x0 ===
Start x0 = [10] -> Converges to x = 28.00, Profit = $78.00k
Start x0 = [38] -> Converges to x = 28.00, Profit = $78.00k
Start x0 = [52] -> Converges to x = 82.00, Profit = $115.00k
Start x0 = [62] -> Converges to x = 82.00, Profit = $115.00k
Start x0 = [90] -> Converges to x = 82.00, Profit = $115.00k

=== ROUND 3A: differential_evolution Global Optimum ===
Global Peak x* = 82.00, Global Max Profit = $115.00k


In [3]:
# ==========================================================================
# ROUND 4 (Q19-Q24): UCR MERCH STORE JACKET PRICING & STORAGE CAP
# ==========================================================================
print('=== ROUND 4: UCR Campus Store Varsity Jacket Scenarios ===')
scenarios = [
    ('Scenario 1: Baseline Wholesale (c=$34, Cap=200)', 130.0, 0.5, 34.0, 200.0),
    ('Scenario 2: Cotton Tariff Shock (c=$52, Cap=200)', 130.0, 0.5, 52.0, 200.0),
    ('Scenario 3: Tariff Shock + Storage Cap (c=$52, Cap=60)', 130.0, 0.5, 52.0, 60.0),
]
for label, p_int, slope, c, cap in scenarios:
    margin = p_int - c
    res = minimize(lambda x: -(margin*x[0] - slope*(x[0]**2)), x0=[20.0], bounds=[(0, cap)])
    q_opt = round(float(res.x[0]), 2)
    p_opt = p_int - slope * q_opt
    pi_opt = -float(res.fun)
    dpi_plus = (margin*(q_opt+1) - slope*((q_opt+1)**2)) - pi_opt
    print(f'{label}: Q* = {q_opt:.0f}, P(Q*) = ${p_opt:.2f}, Profit = ${pi_opt:,.2f}, dPi(+1) = {dpi_plus:+.2f}')

=== ROUND 4: UCR Campus Store Varsity Jacket Scenarios ===
Scenario 1: Baseline Wholesale (c=$34, Cap=200): Q* = 96, P(Q*) = $82.00, Profit = $4,608.00, dPi(+1) = -0.50
Scenario 2: Cotton Tariff Shock (c=$52, Cap=200): Q* = 78, P(Q*) = $91.00, Profit = $3,042.00, dPi(+1) = -0.50
Scenario 3: Tariff Shock + Storage Cap (c=$52, Cap=60): Q* = 60, P(Q*) = $100.00, Profit = $2,880.00, dPi(+1) = +17.50


In [4]:
# ==========================================================================
# ROUND 5 (Q25-Q30): 3-STOCK MARKOWITZ PORTFOLIO OPTIMIZATION
# ==========================================================================
mu = np.array([0.16, 0.11, 0.07])  # Stock 1 (AI Tech 16%), Stock 2 (Biotech 11%), Stock 3 (Utilities 7%)
Sigma = np.array([
    [0.045,  0.004, -0.006],
    [0.004,  0.024,  0.002],
    [-0.006, 0.002,  0.012]
])

def portfolio_variance(w):
    w1, w2, w3 = w
    own_var = 0.045*(w1**2) + 0.024*(w2**2) + 0.012*(w3**2)
    cov_terms = 2*0.004*w1*w2 + 2*(-0.006)*w1*w3 + 2*0.002*w2*w3
    return own_var + cov_terms

def report_portfolio(name, w):
    w = np.array(w)
    w1, w2, w3 = w
    ret = float(w @ mu)
    own_v = float(0.045*(w1**2) + 0.024*(w2**2) + 0.012*(w3**2))
    cov_v = float(2*0.004*w1*w2 + 2*(-0.006)*w1*w3 + 2*0.002*w2*w3)
    tot_v = own_v + cov_v
    std = np.sqrt(tot_v)
    sharpe = (ret - 0.03) / std
    print(f'{name}:')
    print(f'  Weights (w1, w2, w3) = {[round(float(x), 4) for x in w]}')
    print(f'  Expected Return mu_p = {ret*100:.2f}% | own_var = {own_v:.6f} | cov_terms = {cov_v:.6f}')
    print(f'  Total Variance s_p^2 = {tot_v:.6f} | Std Dev s_p = {std*100:.2f}% | Sharpe = {sharpe:.3f}')

print('=== ROUND 5: 3-Stock Markowitz Portfolio Policies ===')
report_portfolio('Policy 1: Equal-Weight (1/3, 1/3, 1/3)', [1/3, 1/3, 1/3])
report_portfolio('Policy 2: Colleague 50/50 Tech-Utilities (0.50, 0.00, 0.50)', [0.50, 0.00, 0.50])

budget_con = LinearConstraint([1, 1, 1], 1.0, 1.0)
ret_con_115 = LinearConstraint(mu, 0.115, np.inf)
res_p3 = minimize(portfolio_variance, x0=[1/3, 1/3, 1/3], bounds=[(0,1)]*3, constraints=[budget_con, ret_con_115], tol=1e-12)
report_portfolio('Policy 3: Optimal Risk-Minimizer (mu_p >= 11.50%)', res_p3.x)

ret_con_095 = LinearConstraint(mu, 0.095, np.inf)
res_p4 = minimize(portfolio_variance, x0=[1/3, 1/3, 1/3], bounds=[(0,1)]*3, constraints=[budget_con, ret_con_095], tol=1e-12)
report_portfolio('Policy 4: Pension Mandate (mu_p >= 9.50% / Global Min Variance)', res_p4.x)

=== ROUND 5: 3-Stock Markowitz Portfolio Policies ===
Policy 1: Equal-Weight (1/3, 1/3, 1/3):
  Weights (w1, w2, w3) = [0.3333, 0.3333, 0.3333]
  Expected Return mu_p = 11.33% | own_var = 0.009000 | cov_terms = -0.000000
  Total Variance s_p^2 = 0.009000 | Std Dev s_p = 9.49% | Sharpe = 0.878
Policy 2: Colleague 50/50 Tech-Utilities (0.50, 0.00, 0.50):
  Weights (w1, w2, w3) = [0.5, 0.0, 0.5]
  Expected Return mu_p = 11.50% | own_var = 0.014250 | cov_terms = -0.003000
  Total Variance s_p^2 = 0.011250 | Std Dev s_p = 10.61% | Sharpe = 0.801
Policy 3: Optimal Risk-Minimizer (mu_p >= 11.50%):
  Weights (w1, w2, w3) = [0.383, 0.2633, 0.3537]
  Expected Return mu_p = 11.50% | own_var = 0.009766 | cov_terms = -0.000446
  Total Variance s_p^2 = 0.009319 | Std Dev s_p = 9.65% | Sharpe = 0.881
Policy 4: Pension Mandate (mu_p >= 9.50% / Global Min Variance):
  Weights (w1, w2, w3) = [0.208, 0.1825, 0.6095]
  Expected Return mu_p = 9.60% | own_var = 0.007204 | cov_terms = -0.000772
  Total Varia